<a href="https://colab.research.google.com/github/CanopySimulations/canopy-python-examples/blob/master/robustness_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import required libraries

In [1]:
# You do not need to run this line everytime. Ensures the runtime supports `asyncio` async/await, and is needed on Google Colab. If the runtime is upgraded, you will be prompted to restart it, which you should do before continuing execution.
# !pip install ipython ipykernel --upgrade
!pip install -q canopy

## Define the required functions and imports

In [92]:
import json
import numpy as np
import asyncio
from typing import List, Dict, Any
import canopy
from canopy import Session, ConfigResult, StudyResult, AuthenticationData
from canopy import load_study
from canopy.openapi import (
    WorksheetApi,
    StudyApi,
    SimVersionApi,
    NewStudyDataSource,
    NewWorksheetDataOutline,
    WorksheetRow,
    WorksheetStudyReference,
    WorksheetRowStudy,
    StudyPostStudyRequest,
    WorksheetConfig,
    WorksheetConfigReference,
    ConfigReferenceTenant,
    WorksheetPutWorksheetRequest,
    WorksheetPostDuplicateConfigsRequest
)

# Define constants to prevent copy-paste errors
STRAIGHT_SIM = "StraightSim"
APEX_SIM = "ApexSim"
DYNAMIC_LAP = "DynamicLap"
DYNAMIC_LAP_STUDY_NAME = "dynamicLap"
SIM_TYPES = [DYNAMIC_LAP]  # Simulations to run
SCALAR_DATA = "scalar_data"
VECTOR_DATA = "vector_data"
STATISTICS = "statistics"
SNR = "snr"
MEASURED = "measured"

n_studies = 5  # Number of studies to run for repeatability

async def authenticate_canopy_sims_api(authentication_data: AuthenticationData) -> Session:
    # Create a session with the authentication data
    session = Session(authentication_data=authentication_data)
    # Try to authenticate the session. If failed, try againwith the same canopy_sims_authentication for upto 10 times.
    for _ in range(10):
        try:
            session.authentication.authenticate()
            print("Authenticated successfully!")
            break
        except Exception as e:
            print(f"Authentication failed: {e}")
            await asyncio.sleep(1)

    return session

async def get_worksheet_from_id(
    session: Session,
    worksheet_id: str,
    worksheet_api: WorksheetApi = None
):
    """
    Retrieves a worksheet by its ID.

    Args:
        session: Authenticated Canopy session object
        worksheet_id: ID of the target worksheet
        worksheet_api: Optional pre-initialized WorksheetApi object (if None, a new one will be created)

    Returns:
        Worksheet object
    """
    # Authenticate and initialize APIs
    session.authentication.authenticate()
    tenant_id = session.authentication.tenant_id
    if worksheet_api is None:
        worksheet_api = WorksheetApi(session.async_client)

    # Get worksheet data
    worksheet_result = await worksheet_api.worksheet_get_worksheet(tenant_id, worksheet_id)
    return worksheet_result.worksheet

async def reset_worksheet(
    session: Session,
    worksheet_id: str,
    row_names: List[str]
):
    """
    Resets a worksheet to contain only specified rows while preserving label definitions.

    Args:
        session: Authenticated Canopy session object
        worksheet_id: ID of worksheet to reset
        row_names: List of row names to preserve in the worksheet

    Returns:
        Updated worksheet object

    Example:
        await reset_worksheet(
            session=active_session,
            worksheet_id="09876087ac087e60da786087d0786ad",
            row_names=["Row1", "Row2"]
        )
    """
    # Authentication and API setup
    worksheet_api = WorksheetApi(session.async_client)
    tenant_id = session.authentication.tenant_id

    # Get current worksheet state
    original_worksheet = await get_worksheet_from_id(session, worksheet_id, worksheet_api)

    # Create filtered outline preserving label definitions
    filtered_outline = NewWorksheetDataOutline(
        rows=[row for row in original_worksheet.outline.rows if row.name in row_names],
        label_definitions=original_worksheet.outline.label_definitions
    )

    # Prepare and execute update
    return await worksheet_api.worksheet_put_worksheet(
        tenant_id,
        worksheet_id,
        WorksheetPutWorksheetRequest(
            name=original_worksheet.name,
            properties=original_worksheet.properties,
            outline=filtered_outline,
            notes=original_worksheet.notes
        )
    )

async def get_all_released_sim_versions(
        session: canopy.Session,
        cut_off_sim_version: str = "1.4292",
        cut_off_sim_version_date: str = "17 December 2020",
        required_sim_versions: list[str] = [],
        excluded_sim_versions: list[str] = [],
        )-> list[dict[str, str]]:
    """Get the simulation versions available in platform from the wiki document.

    Args:
        session: Authenticated Canopy session object
        cut_off_sim_version: Cut-off simulation version to filter the results, default is "1.4292"
        cut_off_sim_version_date: Cut-off simulation version date to filter the results, default is "17 December 2020"
        required_sim_versions: List of required simulation versions to include in the output
        excluded_sim_versions: List of simulation versions to exclude from the output

    Returns:
        List of simulation versions in platform.
    """
    # Authenticate and initialize APIs
    session.authentication.authenticate()
    sim_version_api = SimVersionApi(session.async_client)

    wiki_document = await sim_version_api.sim_version_get_wiki_document(
        wiki_version="current",
        document_path="Release Notes.md"
    )

    content = wiki_document.document.content
    if content is None:
        raise ValueError("No content found in the wiki document for simulation versions.")
    
    # Filter out the versions starting with 1. to a list. Remove the preceeding'##' and the part formatted as 1.xxxxxxxx as the sim_version and the part after the '-'
    # as the date. Create a list of these dicts.
    # Extract the simulation versions from the content
    # The sim_versions are in the format '## 1.xxxxxxxx - date'
    all_sim_versions = [
        {
            "sim_version": line.split(' - ')[0].replace('## ', '').strip(),
            "date": line.split(' - ')[1].strip()
        }
        for line in content.split('\n') if line.startswith('## 1.')
    ]

    cut_off_fraction = int(cut_off_sim_version[2:])

    # First, cut off the versions that are below the cut_off_fraction
    result = []
    for s in all_sim_versions:
        fraction = int(s['sim_version'][2:])
        if fraction < cut_off_fraction:
            break
        result.append(s)

    # Now, filter out the versions that are below the cut_off_sim_version_date. date needs formatting here to be in the format 'DD MMMM YYYY'
    from datetime import datetime
    cut_off_sim_version_date = datetime.strptime(cut_off_sim_version_date, '%d %B %Y').date()
    result = [
        s for s in result if datetime.strptime(s['date'], '%d %B %Y').date() >= cut_off_sim_version_date
    ]

    #Next, make sure that the required simulation versions are included in the result
    if required_sim_versions:
        # If the required simulation versions are not in the result, print a warning
        for required_sim_version in required_sim_versions:
            if required_sim_version not in [s['sim_version'] for s in result]:
                print(f"Warning: Required simulation version '{required_sim_version}' not found in the result.")

    # Finally, remove the excluded simulation versions from the result
    if excluded_sim_versions:
        # Filter the result to exclude the excluded simulation versions
        result = [s for s in result if s['sim_version'] not in excluded_sim_versions]

    # Get the simulation versions
    return result

async def get_all_worksheet_rows(
    session: Session,
    worksheet_id: str
) -> List[WorksheetRow]:
    """Get all worksheet rows in a worksheet.

    Args:
        session: Authenticated Canopy session object
        worksheet_id: ID of the target worksheet

    Returns:
        List of WorksheetRow objects
    """
    # Get worksheet data
    worksheet = await get_worksheet_from_id(session, worksheet_id)

    # Return all worksheet rows
    return worksheet.outline.rows

async def get_worksheet_row_with_name_in_worksheet_with_id(
    session: Session,
    worksheet_id: str,
    worksheet_row_name: str
) -> WorksheetRow:
    """Get worksheet row by name.

    Args:
        session: Authenticated Canopy session object
        worksheet_id: ID of the target worksheet
        worksheet_row_name: Worksheet row name to retrieve

    Returns:
        WorksheetRow object if found, otherwise None
    """
    
    rows = await get_all_worksheet_rows(
        session=session,
        worksheet_id=worksheet_id
    )

    # Get the worksheet row by name
    worksheet_row = next((row for row in rows if row.name == worksheet_row_name), None)

    # Throw an error if the row is not found
    if worksheet_row is None:
        raise ValueError(f"Worksheet row '{worksheet_row_name}' not found in worksheet with ID '{worksheet_id}'.")
    
    return worksheet_row

async def check_worksheet_row_study_exists(
    worksheet_row: WorksheetRow
) -> bool:
    """Check if a worksheet row study exists.

    Args:
        worksheet_row: WorksheetRow object to check

    Returns:
        True if the study exists, False otherwise
    """
    
    # Check if the row has a study reference
    return worksheet_row.study is not None and worksheet_row.study.reference is not None

async def get_study_id_in_worksheet_row(
        session: Session,
        worksheet_id: str,
        worksheet_row_name: str
) -> str:
    """Get the study ID in a worksheet row.

    Args:
        session: Authenticated Canopy session object
        worksheet_id: ID of the target worksheet
        worksheet_row_name: Worksheet row name to analyse
    """
    # Get the worksheet row
    worksheet_row = await get_worksheet_row_with_name_in_worksheet_with_id(
        session=session,
        worksheet_id=worksheet_id,
        worksheet_row_name=worksheet_row_name
    )

    # Check if the worksheet row exists
    b_worksheet_study_exists = await check_worksheet_row_study_exists(worksheet_row)
    if b_worksheet_study_exists is False:
        # Print a message if the study does not exist and return None
        print(f"Worksheet row '{worksheet_row_name}' does not have a study reference.")
        return None
    
    # Get the study ID
    study_id = worksheet_row.study.reference.target_id

    return study_id

async def run_worksheet_row_study_with_configs_having_ids(
    session: Session,
    source_row_name: str,
    sim_version: str,
    worksheet=None,
    row_suffix: str= "",
    new_row_name: str = None,
    config_ids: List[str] = [],
    excluded_config_types: List[str] = [],
    sim_types: List[str] = SIM_TYPES,
    study_type: str = DYNAMIC_LAP_STUDY_NAME,
    notes: str = ""
):
    """Run a study with new configurations in a worksheet row.

    Args:
        session: Authenticated Canopy session object
        worksheet: ID of the target worksheet OR a worksheet object
        source_row_name: Worksheet row name used as the source for configs to be modified
        sim_version: Simulation version
        row_suffix: Suffix to append to modified row names
        config_ids: List of configuration IDs to execute in the new study
        excluded_config_types: List of configuration types to exclude from the new study
        sim_types: List of simulation types to run
        study_type: Type of study to run
        notes: Notes to add to the study
        """

    worksheet_api = WorksheetApi(session.async_client)

    # Make sure that the worksheet exists in some form
    if worksheet is None:
        raise ValueError("Worksheet must either be a worksheet object or worksheet id (str).")
    
    # If we were given a worksheet id instead of a worksheet object, load the worksheet
    if isinstance(worksheet, str):
        worksheet = await get_worksheet_from_id(session, worksheet, worksheet_api)

    # If multiple rows with the same name exist, throw an error
    if len([row for row in worksheet.outline.rows if row.name == source_row_name]) > 1:
        raise ValueError(f"Multiple rows with the name '{source_row_name}' found in worksheet with ID '{worksheet.worksheet_id}'. Please provide a unique row name.")
    
    # Loop through the worksheet rows and get the source row, throw an error if not found
    source_row = next((row for row in worksheet.outline.rows if row.name == source_row_name), None)
    if source_row is None:
        raise ValueError(f"Worksheet row '{source_row_name}' not found in worksheet with ID '{worksheet.worksheet_id}'.")

    # Get the new row name
    if new_row_name is None:
        # If no new row name is provided, use the source row name with the suffix
        new_row_name = f"{source_row.name}{row_suffix}"

    # Create a placeholder for the new configs
    new_configs_list = []

    # Get the duplicate configs if config_ids are provided
    if config_ids:
        target_config_data = await worksheet_api.worksheet_post_duplicate_configs(
            tenant_id=session.authentication.tenant_id,
            worksheet_id=worksheet.worksheet_id,
            worksheet_post_duplicate_configs_request=WorksheetPostDuplicateConfigsRequest(
                source_tenant_id=session.authentication.tenant_id,
                source_worksheet_id=worksheet.worksheet_id,
                source_config_ids=config_ids
                ))
        config_ids = target_config_data.target_config_ids

    # Loop through the config_ids and load the configs
    for config_id in config_ids:
        # Load the new config
        new_config = await canopy.load_config(session, config_id)
        # Create a new worksheet config for the new config
        new_worksheet_config = WorksheetConfig(
            config_type=new_config.document.sub_type,
            reference=WorksheetConfigReference(
                tenant=ConfigReferenceTenant(
                    tenant_id=session.authentication.tenant_id,
                    target_id=config_id
                )
            ),
            inherit_reference=False
        )
        new_configs_list.append(new_worksheet_config)

    # Go through the source row configs and add them if the type is not in any of the config objects in new_configs_list
    for config in source_row.configs:
        # Check if the config type is already in the new configs list
        if config.config_type not in [c.config_type for c in new_configs_list]:
            # Load the source config
            loaded_config = await canopy.load_config(session, config.reference.tenant.target_id)
            # Create a new worksheet config for the source config
            new_worksheet_config = WorksheetConfig(
                config_type=loaded_config.document.sub_type,
                reference=WorksheetConfigReference(
                    tenant=ConfigReferenceTenant(
                        tenant_id=session.authentication.tenant_id,
                        target_id=config.reference.tenant.target_id
                    )
                ),
                inherit_reference=False
            )
            new_configs_list.append(new_worksheet_config)

    # Check if there are no type duplicates in the new configs
    config_types = [config.config_type for config in new_configs_list]
    if len(config_types) != len(set(config_types)):
        raise ValueError(f"Duplicate configuration types found in the new configs: {config_types}.")
    
    # Check if there are any excluded config types in the new configs and remove them
    for config in new_configs_list:
        if config.config_type in excluded_config_types:
            new_configs_list.remove(config)
    
    # Build study configuration
    sources: List[NewStudyDataSource] = []
    
    # Create study configuration
    study = {
        'simTypes': sim_types,
        'simConfig': {}
    }

    # Process non-exploration configs
    for config in new_configs_list:
        loaded_config = await canopy.load_config(session, config.reference.tenant.target_id)
        sources.append(NewStudyDataSource(
            config_type=config.config_type,
            user_id=loaded_config.document.user_id,
            config_id=config.reference.tenant.target_id,
            name=loaded_config.document.name
        ))
        # Explorations are added at the study level; other configs at the simConfig level
        if config.config_type == "exploration":
            # Load the exploration config
            loaded_exploration_config = await canopy.load_config(session, config.reference.tenant.target_id)
            # Add exploration data to the study configuration
            study["exploration"] = loaded_exploration_config.document.data
        else:
            study['simConfig'][config.config_type] = loaded_config.document.data

    # Get the study API
    study_api = StudyApi(session.async_client)

    # Create study
    study_result = await study_api.study_post_study(
        session.authentication.tenant_id,
        StudyPostStudyRequest(
            name=new_row_name,
            study=study,
            is_transient=False,
            study_type=study_type,
            sources=sources,
            notes=notes,
            sim_version=sim_version
        )
    )

    # Create a new row with the new configs and study reference
    new_row = WorksheetRow(
        name=new_row_name,
        configs=new_configs_list,
        study=WorksheetRowStudy(
            reference=WorksheetStudyReference(
                tenant_id=session.authentication.tenant_id,
                target_id=study_result.study_id
            )
        )
    )

    # Update worksheet outline
    worksheet.outline.rows.append(new_row)
    
    # Commit changes
    worksheet_result = await worksheet_api.worksheet_put_worksheet(
        session.authentication.tenant_id,
        worksheet.worksheet_id,
        WorksheetPutWorksheetRequest(
            name=worksheet.name,
            properties=worksheet.properties,
            outline=worksheet.outline,
            notes=worksheet.notes
        )
    )

    # Return the study ID
    return study_result.study_id



## Authenticate

Run this to authenticate yourself with the canopy sims API. There will be a keyboard entry prompt displayed. 

In [4]:
session = await authenticate_canopy_sims_api(canopy.prompt_for_authentication())
user_id = session.authentication.user_id

Authenticated successfully!


# Get the relevant sim versions to test robustness

In [ ]:
cut_off_sim_version = '1.7614'
all_sim_versions_data = await get_all_released_sim_versions(
    session=session,
    cut_off_sim_version=cut_off_sim_version
)

sim_version_list = []

for version_data in all_sim_versions_data:
    sim_version_list.append(version_data['sim_version'])

print(f"Found {len(sim_version_list)} released sim versions after {cut_off_sim_version}:")

latest_sim_version = sim_version_list[0]

# Print the list in one row as comma separated
print(f"{', '.join(reversed(sim_version_list))}")
print(f"Latest sim version is {latest_sim_version}")

# Specify the worksheet details

The worksheet id can be found by inspecting the URL of your worksheet on the browser. It will look something like: 'https://portal.canopysimulations.com/worksheets/zzzzzzzzzzzzzzzzzzzzzzzzzzzzzz/xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx', where the x's represent the worksheet id. Copy this value and set to worksheet_id in the cell below.

In [ ]:
# Put your worksheet ID here
worksheet_id = 'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'
# Enter the worksheet row names of the worksheet from which we extract the car configurations from. These rows will not be modified.
source_row_names = ['Row 1']

# Reset the worksheet

Run this ONLY if you need to clear all the rows except those set in the source_row_names list. Make sure the entries in the list are correct, or else it will irreversibly wipe all your studies!

In [ ]:
# Reset the worksheet to only contain the rows we want to modify
reset_worksheet_result = await reset_worksheet(
    session=session,
    worksheet_id=worksheet_id,
    row_names=source_row_names
)

# Launch studies

The approach below runs an amount of 'n_studies' for each row in the source_row_names, and each new study gets allocated a new row in your worksheet, whose row name gets the row_suffix. 

Although in theory it is possible to launch all the studies at once using an outer for loop, testing revealed that this introduces instability in the compute pool. Hence, it is recommended to do it separately per source row.

The thread sleep also seems to help prevent flaky behaviour when running massive amounts of studies.

In [ ]:
tenant_id = session.authentication.tenant_id
worksheet_api = WorksheetApi(session.async_client)

# Get worksheet data
worksheet_result = await worksheet_api.worksheet_get_worksheet(tenant_id, worksheet_id)
worksheet = worksheet_result.worksheet
sim_version = latest_sim_version
source_row_name = 'Row 1'  # Change this to the desired source row name
for i in range(n_studies):  
    # Create a unique row suffix for each study
    row_suffix = f", n_sim={i+1}"

    print(f"Running study {i+1}/{n_studies} for {source_row_name}")
    
    # Add the new rows to the worksheet
    # Run the worksheet row study with the specified configurations
    study_id = await run_worksheet_row_study_with_configs_having_ids(
        session=session,
        worksheet=worksheet,
        source_row_name=source_row_name,
        sim_version=sim_version,
        row_suffix=row_suffix,
        sim_types=DYNAMIC_LAP,
        study_type=DYNAMIC_LAP,
    )

    # Sleep for a short duration to avoid throttling
    await asyncio.sleep(5)

Running study 1/2 for Baseline
Running study 2/2 for Baseline


# Store the required data locally

We now store the important scalar results such as 'tLapTotal' and 'tDynamicLapQualityMetric' from Dynamic Lap in a JSON file. To identify key elements of robustness such as repeatability, these scalars should reveal any underling issues immediately.

In [94]:
def process_job_data(job_data, required_scalars):
    """
    Processes individual job data and extracts scalar results.
    
    Args:
        job_data: Job data object containing document and scalar data
        required_scalars: The scalar channels you wish to add to the job data
    """
    document = job_data.document
    job_id = job_data.document.document_id
    job_record = {}
    job_record['job_id'] = job_id
    
    # Changes are only relevant if you run an exploration study.
    for change in document.data['changes']:
        # Check if value key exists
        if 'value' in change:
            job_record[change['path']] = change['value']
        # Check if savedConfig exists
        if 'savedConfig' in change:
            job_record[change['path']] = change['savedConfig']['name']
    
    # Get tLapTotal from the data

    job_record[SCALAR_DATA] = {
        scalar_name: job_data.scalar_data.get(scalar_name, None) for scalar_name in required_scalars
    }

    return job_record

async def process_study_row_from_worksheet(row_name, repeatability_sim_data, required_scalars):
    """
    Processes a study row from the worksheet and extracts scalar data.
    
    Args:
        row_name: Name of the worksheet row to process
        source_row_name: Source row name for categorizing data
        required_scalars: The scalar channels you wish to add to the job data
    """
    print(f"Processing row: {row_name}")
    
    # Get the study ID for the row
    study_id = await get_study_id_in_worksheet_row(
        session=session,
        worksheet_id=worksheet_id,
        worksheet_row_name=row_name,
    )

    if not study_id:
        return None
    
    for sim_type in SIM_TYPES:

        if sim_type not in repeatability_sim_data:
            repeatability_sim_data[sim_type] = []

        study = await load_study(
            session=session,
            study_id=study_id,
            sim_type=sim_type,
            include_study_full_document=True,
            include_job_scalar_results=True,
            include_job_full_document=True,
        )
        
        # Get the job data
        if not study or not study.jobs or len(study.jobs) == 0:
            print(f"No jobs found for study {study_id} in row {row_name} for sim type {sim_type}.")
            continue
        
        # Loop through the jobs and get the required scalar data
        for job_data in study.jobs:
            if job_data:
                repeatability_job_data = {
                    "study_id": study_id,
                    **process_job_data(job_data, required_scalars)
                }
                repeatability_sim_data[sim_type].append(repeatability_job_data)

    return repeatability_sim_data

In [102]:
all_worksheet_rows = await get_all_worksheet_rows(
    session=session,
    worksheet_id=worksheet_id,
)

repeatability_study_data = {}
required_scalars = ['tLapTotal', 'tDynamicLapQualityMetric']

# Loop through the rows and collect the study IDs checking if any of the source row names are a part of the worksheet row name
for row in all_worksheet_rows:
    row_name = row.name
    # Check if the row name contains any of the source row names
    for source_row_name in source_row_names:
        # We should check if the source row name is in the row name, but not equal to it
        if source_row_name in row_name and source_row_name != row_name:

            # Lazy initialise
            if source_row_name not in repeatability_study_data:
                repeatability_study_data[source_row_name] = {}

            repeatability_study_data[source_row_name] = await process_study_row_from_worksheet(row_name, repeatability_study_data[source_row_name], required_scalars)

# Save the data on to a file for each outer loop
with open(f'repeatability_study_data.json', 'w') as f:
    json.dump(repeatability_study_data, f, indent=4)

Processing row: Baseline, n_sim=1
Processing row: Baseline, n_sim=2


# Do some statistical post-processing for better robustness analysis

In [103]:
class Stats:
    def __init__(self, mean=None, std_deviation=None, min_val=None, max_val=None):
        self.mean = mean
        self.std_deviation = std_deviation
        self.min = min_val
        self.max = max_val
    
    def to_dict(self):
        return {
            "mean": self.mean,
            "std_deviation": self.std_deviation,
            "min": self.min,
            "max": self.max
        }

def calculate_stats(values) -> Stats:
    """
    Calculates statistical properties of a list of values.

    Args:
        values: A list of numerical values.

    Returns:
        A dictionary with statistical results as standard Python floats.
    """
    if not values:
        return Stats()

    # Convert the input list to a NumPy array to ensure NumPy operations
    # return NumPy scalar types, which have the .item() method.
    np_values = np.array(values, dtype=np.float64)

    # Use .item() to convert the numpy.float64 objects to standard Python floats
    return Stats(
        mean=np_values.mean().item(),
        std_deviation=np_values.std().item(),
        min_val=np_values.min().item(),
        max_val=np_values.max().item()
    )


with open('repeatability_study_data.json', 'r') as f:
    repeatability_study_data = json.load(f)
    statistical_data = []

    # Loop through the dict and calculate the mean and std. deviation of the scalars
    for source_row_name, row_data in repeatability_study_data.items():

        if len(row_data) == 0:
            continue

        # Get the names of the results we're plotting from the first record of the first sim type
        scalar_summary_names = records[0][SCALAR_DATA].keys()

        for key, records in row_data.items():
            if key in SIM_TYPES:
                # Collect up the row name and scalar summary data from all of the sims run related to that row
                row_stats = {
                    'source_row_name': source_row_name,
                    SCALAR_DATA: {
                        scalar_name: [record[SCALAR_DATA].get(scalar_name) for record in records] for scalar_name in scalar_summary_names
                    }
                }
                statistical_data.append(row_stats)

In [106]:
# Now calculate the statistics for each sim version
for data in statistical_data:
    data[STATISTICS] = {scalar_name: calculate_stats(scalar_data) for scalar_name, scalar_data in data[SCALAR_DATA].items()}

In [107]:
for data in statistical_data:
    print(data[STATISTICS]['tLapTotal'].to_dict())
    print(data[STATISTICS]['tDynamicLapQualityMetric'].to_dict())

{'mean': 22.552387500000002, 'std_deviation': 0.0001385000000002634, 'min': 22.552249, 'max': 22.552526}
{'mean': 0.001, 'std_deviation': 0.0, 'min': 0.001, 'max': 0.001}
